# Explore: task B, how long does a user play a game they own?

**Question:** for owned games held out from each user, how well do simple baselines
predict log(1 + minutes), and is within-user rank correlation a usable metric?

Everything comes from the cached `own_*` split (`[split.engagement]`): 3 owned games
held out per user for validation and 3 for test, users with ≥ 7 owned games. Never-played
games stay in (target 0). Validation only; test stays untouched.

Metrics:
- **MSE / MAE** on y = log(1 + minutes)
- **within-user Spearman** over each user's 3 held-out games: does the model put a user's
  games in the right order? Undefined when a user's 3 targets are all equal.

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from assignment_2.modules.config import get_config
from assignment_2.modules.core import (
    calc_mse,
    compare,
    evaluate_engagement,
    within_user_spearman,
)
from assignment_2.modules.data import load_prepped

get_config.cache_clear()
cfg = get_config()
t = load_prepped(cfg)
train, valid = t["own_train"], t["own_valid"]


def target(df: pd.DataFrame) -> np.ndarray:
    return np.log1p(df.playtime_forever.to_numpy(float))


y_train, y_valid = target(train), target(valid)
print(f"sample={cfg.sample.tag}  train {len(train):,} rows, valid {len(valid):,} rows, "
      f"{valid.user_id.nunique():,} valid users")

sample=users_0.1  train 454,741 rows, valid 18,034 rows, 6,020 valid users


## 1. Is the holdout representative?

Held-out games are drawn at random within each user, but every eligible user gives exactly
3, so valid/test weight users equally, while train rows are dominated by users with huge
libraries. Compare train *weighted by user* with valid: a big difference would mean the
split is biased (or a bug).

In [2]:
def describe_target(y: np.ndarray) -> dict:
    return {
        "rows": f"{len(y):,}",
        "never played (y = 0)": f"{(y == 0).mean():.1%}",
        "median y": round(float(np.median(y)), 2),
        "mean y": round(float(y.mean()), 2),
        "median hours (played only)": round(float(np.median(np.expm1(y[y > 0])) / 60), 1),
    }


# Held-out rows are 3 per user, so valid/test weight users equally; train rows are
# dominated by users with huge libraries. Weight train by user (same users as valid)
# to compare like with like.
zero = pd.Series(y_train == 0, index=train.index)
in_valid = train.user_id.isin(valid.user_id)
user_weighted = zero[in_valid].groupby(train.user_id[in_valid]).mean().mean()

table = pd.DataFrame({
    "train": describe_target(y_train),
    "valid": describe_target(y_valid),
    "test": describe_target(target(t["own_test"])),
})
table.loc["never played, user-weighted"] = [f"{user_weighted:.1%}", "", ""]
table

,train,valid,test
rows,"454,741","18,034","18,041"
never played (y = 0),36.3%,31.0%,30.7%
median y,3.53,4.06,4.13
mean y,3.34,3.74,3.78
median hours (played only),3.4,4.0,4.1
"never played, user-weighted",30.8%,,


## 2. Can within-user Spearman be computed?

With 3 held-out games per user, ties are common: many owned games have 0 minutes. A user
whose 3 games are *all* tied has no defined rank correlation, so the metric skips them.
How many users is that, and is it a problem?

In [3]:
v = valid.assign(y=y_valid)
ties = v.groupby("user_id").y.agg(
    n="size",
    distinct="nunique",
    all_zero=lambda s: (s == 0).all(),
)
# drop_cold can remove a held-out game whose item never appears in train, so a few
# users have fewer than 3 (counts below).
assert (ties.n <= cfg.split.engagement.n_holdout).all()
print(ties.n.value_counts().sort_index(ascending=False).to_dict())

pd.Series({
    "valid users": f"{len(ties):,}",
    "all 3 never played": f"{ties.all_zero.mean():.1%}",
    "all 3 tied (Spearman undefined)": f"{(ties.distinct == 1).mean():.1%}",
    "some tie (2 distinct values)": f"{(ties.distinct == 2).mean():.1%}",
    "no ties": f"{(ties.distinct == 3).mean():.1%}",
}, name="value").to_frame()

{3: 5995, 2: 24, 1: 1}


,value
valid users,"6,020"
all 3 never played,5.8%
all 3 tied (Spearman undefined),5.8%
some tie (2 distinct values),19.8%
no ties,74.4%


## 3. Metrics

`evaluate_engagement` (`core/evaluate.py`) computes MSE, MAE and within-user Spearman
**per user**, then averages them, so every user counts equally (as the protocol says),
with a 95% bootstrap interval for each. `within_user_spearman` (`core/metrics.py`) is
the Pearson correlation of within-user ranks (ties get their average rank, as in
`scipy.stats.spearmanr`), NaN for users where it's undefined; those are skipped.

In [4]:
N_BOOT = 1000


def evaluate(pred: np.ndarray) -> pd.Series:
    """Per-user-averaged metrics on own_valid, as one table row."""
    summary, per_user = evaluate_engagement(valid, pred, n_boot=N_BOOT, seed=cfg.seed)
    return pd.Series({
        "mse": summary["mse"].value,
        "mae": summary["mae"].value,
        "spearman (mean)": summary["spearman"].value,
        "spearman users": per_user["spearman"].notna().mean(),  # share where it's defined
    })


# sanity checks: perfect predictions score rho = 1; reversed ones rho = -1
assert np.isclose(evaluate(y_valid)["spearman (mean)"], 1.0)
assert np.isclose(evaluate(-y_valid)["spearman (mean)"], -1.0)

**Note:** "spearman users" is the share of users where ρ is defined for *that model*. A
model that predicts the same value for all 3 of a user's games (global mean, user mean)
has no ranking at all, so ρ is undefined for everyone: those baselines are scored on MSE
and MAE only.

## 4. Baselines

Each baseline fits on `own_train` and returns one prediction per `valid` row.

- **global mean:** μ for everyone (the MSE floor every model must beat)
- **user mean:** how much this user plays their games in general
- **item mean:** how much owners play this game in general
- **bias model:** μ + b_u + b_i, fit by alternating regularized means (the LFM without
  the γ_u · γ_i term, trained on owned games only, as task B's model will be)

In [5]:
mu = y_train.mean()
tr = train.assign(y=y_train)


def global_mean() -> np.ndarray:
    return np.full(len(valid), mu)


def user_mean(lam: float = 5.0) -> np.ndarray:
    """User's mean target, shrunk toward μ with `lam` pseudo-games."""
    g = tr.groupby("user_id").y
    b = (g.sum() - g.size() * mu) / (g.size() + lam)
    return mu + valid.user_id.map(b).fillna(0).to_numpy()


def item_mean(lam: float = 5.0) -> np.ndarray:
    """Game's mean target among train owners, shrunk toward μ."""
    g = tr.groupby("item_id").y
    b = (g.sum() - g.size() * mu) / (g.size() + lam)
    return mu + valid.item_id.map(b).fillna(0).to_numpy()


def bias_model(lam_u: float = 5.0, lam_i: float = 5.0, n_iter: int = 10) -> np.ndarray:
    """μ + b_u + b_i, alternating ridge updates (closed form for each block)."""
    r = tr.y - mu
    b_i = pd.Series(0.0, index=tr.item_id.unique())
    for _ in range(n_iter):
        res_u = r - tr.item_id.map(b_i)
        b_u = res_u.groupby(tr.user_id).sum() / (tr.groupby("user_id").size() + lam_u)
        res_i = r - tr.user_id.map(b_u)
        b_i = res_i.groupby(tr.item_id).sum() / (tr.groupby("item_id").size() + lam_i)
    return (mu + valid.user_id.map(b_u).fillna(0) + valid.item_id.map(b_i).fillna(0)).to_numpy()

### Shrinkage
Pick `lam` on validation for the item mean and the bias model (a small grid is enough;
results are usually flat).

In [6]:
grid = [1, 5, 10, 25, 50]
pd.DataFrame({
    "item mean": {lam: calc_mse(y_valid, item_mean(lam)) for lam in grid},
    "bias model": {lam: calc_mse(y_valid, bias_model(lam, lam)) for lam in grid},
}).rename_axis("lam").round(4)

,item mean,bias model
lam,,
1,5.5869,5.0774
5,5.5940,5.0289
10,5.6134,5.0615
25,5.6751,5.1760
50,5.7658,5.3283


In [7]:
LAM_ITEM, LAM_BIAS = 1, 5   # best validation MSE in the grid above

results = pd.DataFrame({
    "global mean": evaluate(global_mean()),
    "user mean": evaluate(user_mean(LAM_BIAS)),
    "item mean": evaluate(item_mean(LAM_ITEM)),
    "bias model": evaluate(bias_model(LAM_BIAS, LAM_BIAS)),
}).T.round(4)
results

,mse,mae,spearman (mean),spearman users
global mean,9.8904,2.7840,NaN,0.0000
user mean,8.9015,2.5663,NaN,0.0000
item mean,5.5852,1.8710,0.5361,0.9417
bias model,5.0274,1.7812,0.5404,0.9417


### Two-part (hurdle) check
31% of held-out targets are 0. Does splitting "played at all?" from "how long, if played?"
help at the item level?

In [8]:
def item_hurdle(lam: float = 5.0) -> np.ndarray:
    """P(played | item) × E[y | played, item], both shrunk toward their overall values."""
    played = tr.y > 0
    owners = tr.groupby("item_id").size()
    p0 = played.mean()
    p = (played.groupby(tr.item_id).sum() + lam * p0) / (owners + lam)

    pos = tr[played]
    m0 = pos.y.mean()
    g = pos.groupby("item_id").y
    m = ((g.sum().reindex(owners.index, fill_value=0) + lam * m0)
         / (g.size().reindex(owners.index, fill_value=0) + lam))
    return (valid.item_id.map(p).fillna(p0) * valid.item_id.map(m).fillna(m0)).to_numpy()


pd.DataFrame({
    "item mean": evaluate(item_mean(LAM_ITEM)),
    "item hurdle": evaluate(item_hurdle(LAM_ITEM)),
}).T.round(4)

,mse,mae,spearman (mean),spearman users
item mean,5.5852,1.8710,0.5361,0.9417
item hurdle,5.5848,1.8714,0.5357,0.9417


### Are the differences real?
Paired bootstrap over users (`compare`): Δ = first − second, with a 95% interval. For MSE
lower is better, so a negative Δ whose interval excludes 0 means the first model is
better.

In [9]:
preds = {
    "global mean": global_mean(),
    "user mean": user_mean(LAM_BIAS),
    "item mean": item_mean(LAM_ITEM),
    "bias model": bias_model(LAM_BIAS, LAM_BIAS),
}
pu = {name: evaluate_engagement(valid, p, n_boot=N_BOOT, seed=cfg.seed)[1]
      for name, p in preds.items()}

pairs = [
    ("user mean", "global mean", "mse"),
    ("item mean", "user mean", "mse"),
    ("bias model", "item mean", "mse"),
    ("bias model", "item mean", "spearman"),   # b_u can't reorder a user's games
]
rows = {}
for a, b, metric in pairs:
    d = compare(pu[a][metric], pu[b][metric], n_boot=N_BOOT, seed=cfg.seed)
    rows[(f"{a} − {b}", metric)] = {"difference": d.value, "ci low": d.ci_low,
                                     "ci high": d.ci_high, "real?": d.ci_low > 0 or d.ci_high < 0}
pd.DataFrame(rows).T.round(4)

difference    ci low   ci high  real?
user mean − global mean mse        -0.98888 -1.077678 -0.905611   True
item mean − user mean   mse       -3.316292 -3.467958 -3.160144   True
bias model − item mean  mse       -0.557784 -0.612222  -0.50335   True
                        spearman   0.004315 -0.000329  0.009402  False

## 5. Does popularity order a user's games?

Task A's models output ranking scores, not hours. The cross-task test ("BPR on task B")
scores them on within-user Spearman only. Owner popularity is the simplest such score:
how well does "more owners" predict "played longer", *within* a user?

In [10]:
owners = train.item_id.value_counts()
pop = valid.item_id.map(owners).fillna(0).to_numpy(float)
rho_pop = within_user_spearman(valid.user_id, y_valid, pop)
print(f"owner popularity: within-user Spearman {rho_pop.mean():.3f} "
      f"(defined for {rho_pop.notna().mean():.1%} of users)")

owner popularity: within-user Spearman 0.165 (defined for 94.2% of users)


## 6. Questions to answer
- Does the holdout look like train (§1)? If the zero shares differ by more than a point or
  two, check the split.
- What share of users have no defined Spearman (§2)? If it's large, report Spearman over
  the users where it's defined, say how many that is, and lean on MSE / MAE.
- Is the zero mass a problem for squared error? If ~36% of targets are 0, a two-part
  (hurdle) model, P(played) × E[log minutes | played], may beat a single regression.
- How much does the bias model gain over the item mean? The gap left above the bias model
  is the room for γ_u · γ_i in the LFM.
- Does popularity order a user's games (§5)? That's the bar for the cross-task test.

**Takeaways:**
- 99.6% of validation users keep all 3 held-out games; 25 lost one or two to cold drops.
- Held-out rows are user-weighted (3 per user) while train rows are dominated by large
  libraries, so valid looks more played than train (31.0% vs 36.3% never played). Weighted
  by user, train matches: 30.8%. The split is representative. → Task B's metrics weight
  every user equally; say so in the write-up.
- Within-user Spearman is defined for 94% of users (the rest have all 3 held-out games
  tied, nearly always all 0 minutes). → Report mean ρ over those users.
- The game matters far more than the user (MSE averaged per user): 9.89 (global mean) →
  8.90 (user mean) → 5.59 (item mean, λ = 1) → 5.03 (bias model, λ = 5). Each step is a
  real improvement (paired bootstrap, ΔMSE 95% CI): user − global −0.99 [−1.08, −0.91],
  item − user −3.32 [−3.47, −3.16], bias − item −0.56 [−0.61, −0.50].
- Within a user, b_u is constant, so the bias model orders a user's games like the item
  mean: Spearman 0.540 vs 0.536, Δ +0.004 [−0.000, +0.009], not a real difference. Only
  γ_u · γ_i can reorder a user's games, so Spearman measures what the LFM adds beyond
  biases.
- An item-level hurdle matches the item mean (MSE 5.586 vs 5.587): splitting "played?" from
  "how long?" only helps once P(played) depends on the user. → Deferred to the model stage.